# KAN Reusability Experiments

Three focused experiments probing when transferring `BaseKANModel.kan` weights
between problems actually pays off. Each section is self-contained — run the
**Setup** cell once, then any section in any order. Config variables sit at
the top of each section cell; edit and re-run.

1. **Same-density resolution transfer** — the cleanest test: source and target
   share physics, density, and optimal design; only the FEM mesh is 2x finer.
2. **Pre-train length sweep** — tests the saturation hypothesis: does a
   half-converged (still gray) source transfer better than a fully converged
   crisp one?
3. **Bigger targets + multiple seeds** — 4x targets where scratch training is
   genuinely expensive, with seed-averaged results so conclusions aren't noise.


In [ ]:
# --- Setup: run once per kernel session ---
import copy
import time

import numpy as np
import torch
import matplotlib.pyplot as plt

from neural_structural_optimization import problems, topo_api
import models as pt


def build_kan(problem_name, seed=0, kan_layers=(16, 16), grid=8, k=3):
    args = topo_api.specified_task(problems.PROBLEMS_BY_NAME[problem_name])
    return pt.BaseKANModel(seed=seed, args=args, kan_layers=kan_layers,
                            grid=grid, k=k)


def train_timed(model, steps, progress_every=None):
    t0 = time.time()
    ds = pt.train_lbfgs(model, steps, progress_every=progress_every)
    elapsed = time.time() - t0
    losses = ds.loss.values
    best_step = int(np.nanargmin(losses))
    design = np.clip(ds.design.isel(step=best_step).values, 0.0, 1.0)
    return float(np.nanmin(losses)), design, losses, elapsed


def transfer_weights(src_model, tgt_model):
    tgt_model.kan.load_state_dict(copy.deepcopy(src_model.kan.state_dict()))
    return tgt_model


def zero_shot_eval(model):
    with torch.no_grad():
        logits = model()
        compliance = float(model.loss(logits).item())
        design = model.env.render(
            logits.detach().cpu().numpy().reshape(-1), volume_contraint=True)
    return compliance, np.clip(design, 0.0, 1.0)


def crossover(losses, total_time, bar):
    """First (step, ~seconds) where losses <= bar, else (None, None).
    Per-step times approximated by spreading total_time uniformly."""
    hits = np.flatnonzero(np.asarray(losses) <= bar)
    if hits.size == 0:
        return None, None
    step = int(hits[0])
    return step, total_time * (step + 1) / len(losses)


def show_designs(items, suptitle=None):
    """items: list of (title, 2D design array)."""
    fig, axes = plt.subplots(1, len(items), figsize=(5.5 * len(items), 3.4))
    if len(items) == 1:
        axes = [axes]
    for ax, (label, design) in zip(axes, items):
        ax.imshow(1.0 - design, cmap="gray", vmin=0.0, vmax=1.0)
        ax.set_title(label, fontsize=10)
        ax.axis("off")
    if suptitle:
        fig.suptitle(suptitle)
    plt.tight_layout()
    plt.show()


print(f"Setup OK — {len(problems.PROBLEMS_BY_NAME)} problems available "
      f"(incl. same-density scalings: mbb_beam_192x64_0.5, mbb_beam_384x128_0.5, "
      f"cantilever_beam_full_192x64_0.4, cantilever_beam_full_384x128_0.4)")

## Section 1 — Same-density resolution transfer (exact 2x grid)

`mbb_beam_96x32_0.5 -> mbb_beam_192x64_0.5`: identical physics, identical
density, exactly twice the mesh in each direction. The KAN's learned density
function phi(x, y) lives on normalized coordinates, so if resolution
independence is real, the transferred weights should be a near-perfect warm
start here — this pair removes every confound the earlier
`0.5 -> 0.4` default had.

In [ ]:
# --- Section 1 config ---
SOURCE = "mbb_beam_96x32_0.5"
TARGET = "mbb_beam_192x64_0.5"   # same density, exactly 2x grid
PRETRAIN_STEPS = 100
TARGET_STEPS = 300
KAN_LAYERS, GRID, K = (16, 16), 8, 3
SEED = 0

print(f"[pretrain] {SOURCE}, {PRETRAIN_STEPS} steps...")
src = build_kan(SOURCE, SEED, KAN_LAYERS, GRID, K)
_, _, _, pre_t = train_timed(src, PRETRAIN_STEPS)
print(f"  {pre_t:.2f}s (sunk cost, paid once)")

print(f"[scratch] {TARGET}, {TARGET_STEPS} steps...")
scratch = build_kan(TARGET, SEED + 1, KAN_LAYERS, GRID, K)
s_comp, s_design, s_losses, s_t = train_timed(scratch, TARGET_STEPS, progress_every=100)
print(f"  compliance={s_comp:.4f} in {s_t:.2f}s")

zs = transfer_weights(src, build_kan(TARGET, SEED + 1, KAN_LAYERS, GRID, K))
z_comp, z_design = zero_shot_eval(zs)
print(f"[zero-shot] compliance={z_comp:.4f} (0s of target training)")

print(f"[transfer] fine-tuning {TARGET_STEPS} steps...")
ft = transfer_weights(src, build_kan(TARGET, SEED + 1, KAN_LAYERS, GRID, K))
f_comp, f_design, f_losses, f_t = train_timed(ft, TARGET_STEPS, progress_every=100)
cross_step, cross_t = crossover(f_losses, f_t, s_comp)
print(f"  compliance={f_comp:.4f} in {f_t:.2f}s")

print()
if cross_step is not None:
    saved = s_t - cross_t
    print(f"RESULT: transfer matched scratch quality ({s_comp:.4f}) at step "
          f"{cross_step + 1}/{len(f_losses)} (~{cross_t:.1f}s vs {s_t:.1f}s scratch)")
    print(f"        => ~{saved:.1f}s saved ({100 * saved / s_t:.1f}% of scratch time)")
else:
    print(f"RESULT: transfer never matched scratch ({s_comp:.4f}); best was {f_comp:.4f}")

show_designs([
    (f"from scratch\ncompliance={s_comp:.2f}, {s_t:.1f}s", s_design),
    (f"zero-shot transfer\ncompliance={z_comp:.2f}, 0s", z_design),
    (f"fine-tuned transfer\ncompliance={f_comp:.2f}, {f_t:.1f}s", f_design),
], suptitle=f"{SOURCE} -> {TARGET}")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(np.linspace(s_t / len(s_losses), s_t, len(s_losses)), s_losses, label="from scratch")
ax.plot(np.linspace(f_t / len(f_losses), f_t, len(f_losses)), f_losses, label="fine-tuned transfer")
ax.axhline(s_comp, color="black", linestyle=":", lw=1, label="scratch final compliance")
ax.axhline(z_comp, color="gray", linestyle="--", lw=1, label="zero-shot")
if cross_t is not None:
    ax.axvline(cross_t, color="red", linestyle="--", lw=1, label=f"crossover (~{cross_t:.1f}s)")
ax.set_xlabel("Target training time (s, approximate)")
ax.set_ylabel("Compliance")
ax.set_yscale("log")
ax.legend(fontsize=9); ax.grid(True, alpha=0.4)
plt.tight_layout(); plt.show()

## Section 2 — Pre-train length sweep (saturation hypothesis)

If the source design converges to a crisp 0/1 layout, the sigmoid saturates
and gradients through the transferred weights nearly vanish — so a
*half-converged* source (structure visible, still gray) may transfer better
than a fully converged one. This sweep pre-trains the source for increasing
step counts and measures the transfer benefit of each.

In [ ]:
# --- Section 2 config ---
SOURCE = "mbb_beam_96x32_0.5"
TARGET = "mbb_beam_192x64_0.5"
PRETRAIN_SWEEP = [25, 50, 100, 200, 400]
TARGET_STEPS = 300
KAN_LAYERS, GRID, K = (16, 16), 8, 3
SEED = 0

# The scratch bar is shared by every sweep point — train it once.
print(f"[scratch] {TARGET}, {TARGET_STEPS} steps (shared quality bar)...")
scratch = build_kan(TARGET, SEED + 1, KAN_LAYERS, GRID, K)
s_comp, _, _, s_t = train_timed(scratch, TARGET_STEPS)
print(f"  compliance={s_comp:.4f} in {s_t:.2f}s")

rows = []
for n_pre in PRETRAIN_SWEEP:
    src = build_kan(SOURCE, SEED, KAN_LAYERS, GRID, K)
    _, _, _, pre_t = train_timed(src, n_pre)

    zs = transfer_weights(src, build_kan(TARGET, SEED + 1, KAN_LAYERS, GRID, K))
    z_comp, _ = zero_shot_eval(zs)

    ft = transfer_weights(src, build_kan(TARGET, SEED + 1, KAN_LAYERS, GRID, K))
    f_comp, _, f_losses, f_t = train_timed(ft, TARGET_STEPS)
    cross_step, cross_t = crossover(f_losses, f_t, s_comp)

    rows.append((n_pre, pre_t, z_comp, f_comp, cross_step, cross_t))
    cross_txt = (f"step {cross_step + 1} (~{cross_t:.1f}s)"
                 if cross_step is not None else "never")
    print(f"pretrain={n_pre:>4}: zero-shot={z_comp:9.3f}  fine-tuned={f_comp:9.3f}"
          f"  crossover={cross_txt}")

print()
print(f"{'pretrain':>8} {'zero-shot':>11} {'fine-tuned':>11} {'crossover step':>15} {'~time saved':>12}")
for n_pre, pre_t, z_comp, f_comp, cross_step, cross_t in rows:
    saved = f"{s_t - cross_t:8.1f}s" if cross_t is not None else "     n/a"
    step_txt = str(cross_step + 1) if cross_step is not None else "never"
    print(f"{n_pre:>8} {z_comp:>11.3f} {f_comp:>11.3f} {step_txt:>15} {saved:>12}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
pre_ns = [r[0] for r in rows]
ax1.plot(pre_ns, [r[2] for r in rows], "o-", label="zero-shot compliance")
ax1.plot(pre_ns, [r[3] for r in rows], "s-", label="fine-tuned compliance")
ax1.axhline(s_comp, color="black", linestyle=":", lw=1, label="scratch bar")
ax1.set_xlabel("Source pre-train steps"); ax1.set_ylabel("Compliance")
ax1.set_xscale("log"); ax1.legend(fontsize=9); ax1.grid(True, alpha=0.4)
cross_steps = [r[4] + 1 if r[4] is not None else np.nan for r in rows]
ax2.plot(pre_ns, cross_steps, "o-")
ax2.set_xlabel("Source pre-train steps")
ax2.set_ylabel(f"Fine-tune steps to match scratch (of {TARGET_STEPS})")
ax2.set_xscale("log"); ax2.grid(True, alpha=0.4)
plt.tight_layout(); plt.show()

## Section 3 — Bigger targets, multiple seeds

At 192x64 the scratch run costs seconds, so savings drown in noise. Here the
target is the 4x-scaled `mbb_beam_384x128_0.5` (same density), and every
condition runs across several seeds so the comparison reports mean +- std.
Expect this cell to take several minutes — it is the closest analogue to the
HPC amortization script (`5.3_neural_reuse/amortization_experiment.py`).

In [ ]:
# --- Section 3 config ---
SOURCE = "mbb_beam_96x32_0.5"
TARGET = "mbb_beam_384x128_0.5"   # same density, exactly 4x grid
PRETRAIN_STEPS = 100
TARGET_STEPS = 300
SEEDS = [0, 1, 2]
KAN_LAYERS, GRID, K = (16, 16), 8, 3

results = {"scratch_comp": [], "scratch_t": [], "finetune_comp": [],
           "finetune_t": [], "cross_t": [], "zeroshot_comp": []}
last_designs = {}

for seed in SEEDS:
    print(f"=== seed {seed} ===")
    src = build_kan(SOURCE, seed, KAN_LAYERS, GRID, K)
    _, _, _, pre_t = train_timed(src, PRETRAIN_STEPS)

    scratch = build_kan(TARGET, seed + 100, KAN_LAYERS, GRID, K)
    s_comp, s_design, s_losses, s_t = train_timed(scratch, TARGET_STEPS, progress_every=100)
    print(f"  scratch:   compliance={s_comp:.4f} in {s_t:.1f}s")

    zs = transfer_weights(src, build_kan(TARGET, seed + 100, KAN_LAYERS, GRID, K))
    z_comp, _ = zero_shot_eval(zs)

    ft = transfer_weights(src, build_kan(TARGET, seed + 100, KAN_LAYERS, GRID, K))
    f_comp, f_design, f_losses, f_t = train_timed(ft, TARGET_STEPS, progress_every=100)
    cross_step, cross_t = crossover(f_losses, f_t, s_comp)
    print(f"  transfer:  compliance={f_comp:.4f} in {f_t:.1f}s, "
          + (f"crossover ~{cross_t:.1f}s" if cross_t is not None else "no crossover"))

    results["scratch_comp"].append(s_comp)
    results["scratch_t"].append(s_t)
    results["finetune_comp"].append(f_comp)
    results["finetune_t"].append(f_t)
    results["cross_t"].append(cross_t if cross_t is not None else np.nan)
    results["zeroshot_comp"].append(z_comp)
    last_designs = {"scratch": (s_comp, s_design), "finetune": (f_comp, f_design)}


def ms(vals):
    vals = np.asarray(vals, dtype=float)
    return f"{np.nanmean(vals):.3f} +- {np.nanstd(vals):.3f}"

print()
print(f"{TARGET}, {len(SEEDS)} seeds:")
print(f"  scratch compliance:    {ms(results['scratch_comp'])}   time {ms(results['scratch_t'])}s")
print(f"  zero-shot compliance:  {ms(results['zeroshot_comp'])}")
print(f"  fine-tune compliance:  {ms(results['finetune_comp'])}   time {ms(results['finetune_t'])}s")
n_crossed = int(np.sum(~np.isnan(results['cross_t'])))
print(f"  crossover reached in {n_crossed}/{len(SEEDS)} seeds"
      + (f", mean ~{np.nanmean(results['cross_t']):.1f}s vs "
         f"{np.mean(results['scratch_t']):.1f}s scratch" if n_crossed else ""))

show_designs([
    (f"from scratch (last seed)\ncompliance={last_designs['scratch'][0]:.2f}",
     last_designs['scratch'][1]),
    (f"fine-tuned transfer (last seed)\ncompliance={last_designs['finetune'][0]:.2f}",
     last_designs['finetune'][1]),
], suptitle=f"{SOURCE} -> {TARGET}")